# Common Preprocessing — Kosovo Investment Analysis

Data from ATK investment reports, 2019–2026. The last year covers January–June only. Each row represents an aggregated group, not an individual taxpayer.

**Run first:** `python -m phase_1.run_common_preprocessing` from the project root. This notebook reviews the saved result; it does not re-run or modify the preprocessing pipeline.

## 1. Load the processed data

In [1]:
from pathlib import Path
import sys
import json
import pandas as pd
from IPython.display import display

root = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "src/schema.py").exists()), None)
if root is None:
    raise FileNotFoundError("Open the notebook inside the project folder.")
sys.path.insert(0, str(root))

from src.schema import SOURCE_COLUMNS, MONEY, CATEGORIES, BUSINESS_KEY
from src.io_utils import load_dataset

path = root / "data/processed/common_dataset.csv"
if not path.exists():
    raise FileNotFoundError("Run: python -m phase_1.run_common_preprocessing")
df = load_dataset(path)
print(f"Records: {len(df):,} | Source attributes: {len(SOURCE_COLUMNS)}")
display(df[SOURCE_COLUMNS].head(5))

Records: 124,262 | Source attributes: 12


,year,month,primary_sector,registration_status,municipality,taxpayer_count,investment_purchases_imports_without_vat,investment_purchases_imports_nondeductible_vat,investment_imports_18pct,investment_imports_8pct,domestic_investment_purchases_18pct,domestic_investment_purchases_8pct
0,2019,1,Administrimi publik dhe mbrojtja; Sigurimi soc...,SH.P.K.,FERIZAJ,5,0.0,0.0,0.0,0.0,0.0,0.0
1,2019,1,Administrimi publik dhe mbrojtja; Sigurimi soc...,Individual,FUSHË KOSOVË,1,0.0,0.0,0.0,0.0,0.0,0.0
2,2019,1,Administrimi publik dhe mbrojtja; Sigurimi soc...,SH.P.K.,FUSHË KOSOVË,3,4000.0,0.0,0.0,0.0,0.0,0.0
3,2019,1,Administrimi publik dhe mbrojtja; Sigurimi soc...,OJQ,GJAKOVË,1,0.0,0.0,0.0,0.0,0.0,0.0
4,2019,1,Administrimi publik dhe mbrojtja; Sigurimi soc...,SH.P.K.,GJAKOVË,3,0.0,0.0,0.0,0.0,0.0,0.0


## 2. Structure and attribute types

In [2]:
df[SOURCE_COLUMNS].info()
display(df[SOURCE_COLUMNS].describe(include="all").T.round(2))

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 124262 entries, 0 to 124261
Data columns (total 12 columns):
 #   Column                                          Non-Null Count   Dtype  
---  ------                                          --------------   -----  
 0   year                                            124262 non-null  Int16  
 1   month                                           124262 non-null  Int8   
 2   primary_sector                                  124262 non-null  string 
 3   registration_status                             124262 non-null  string 
 4   municipality                                    124262 non-null  string 
 5   taxpayer_count                                  124262 non-null  Int64  
 6   investment_purchases_imports_without_vat        124262 non-null  float64
 7   investment_purchases_imports_nondeductible_vat  124262 non-null  float64
 8   investment_imports_18pct                        124262 non-null  float64
 9   investment_imports_8pct   

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
year,124262.0,<NA>,<NA>,<NA>,2022.315334,2.187246,2019.0,2020.0,2022.0,2024.0,2026.0
month,124262.0,<NA>,<NA>,<NA>,6.289791,3.445988,1.0,3.0,6.0,9.0,12.0
primary_sector,124262,23,Industria perpunuese,11320,NaN,NaN,NaN,NaN,NaN,NaN,NaN
registration_status,124262,26,SH.P.K.,45991,NaN,NaN,NaN,NaN,NaN,NaN,NaN
municipality,124262,38,PRISHTINË,11388,NaN,NaN,NaN,NaN,NaN,NaN,NaN
taxpayer_count,124262.0,<NA>,<NA>,<NA>,21.021954,90.203245,1.0,1.0,3.0,11.0,2992.0
investment_purchases_imports_without_vat,124262.0,NaN,NaN,NaN,23947.680539,413233.93426,-952577.08,0.0,0.0,0.0,95609575.76
investment_purchases_imports_nondeductible_vat,124262.0,NaN,NaN,NaN,8116.002767,88407.588467,-664578.8,0.0,0.0,0.0,9246564.34
investment_imports_18pct,124262.0,NaN,NaN,NaN,8177.079446,66223.572793,-68173.32,0.0,0.0,0.0,8817568.38
investment_imports_8pct,124262.0,NaN,NaN,NaN,248.014816,10282.029141,0.0,0.0,0.0,0.0,2111097.6


`year` and `month` identify the reporting period; sector, registration status and municipality are categorical. `taxpayer_count` is a count, while the six VAT-investment columns are numerical amounts. The six amounts should not automatically be added together.

## 3. Reporting coverage and integration

In [3]:
year_counts = df.groupby("year", observed=True).size().rename("records")
display(year_counts.rename_axis("year").reset_index())
periods = df.groupby("year", observed=True)["month"].nunique().rename("months")
display(periods.rename_axis("year").reset_index())
print("Original files:", df["source_file"].nunique())

,year,records
0,2019,16395
1,2020,16307
2,2021,15949
3,2022,16201
4,2023,16332
5,2024,17249
6,2025,17214
7,2026,8615


,year,months
0,2019,12
1,2020,12
2,2021,12
3,2022,12
4,2023,12
5,2024,12
6,2025,12
7,2026,6


Original files: 8


The annual Excel reports were standardized and combined by the processing scripts. The coverage for 2026 is partial, so full-year comparisons should exclude it or compare January–June across years.

## 4. Data quality: missing values and duplicates

In [4]:
nulls = df[SOURCE_COLUMNS].isna().sum().rename("nulls").reset_index()
nulls.columns = ["attribute", "nulls"]
display(nulls)

unclassified = df["primary_sector"].eq("Mungon aktiviteti")
print(f"Unclassified sector: {unclassified.sum():,} ({unclassified.mean():.2%})")
print("Duplicate source rows:", int(df.duplicated(subset=["source_file", "source_sheet", "source_excel_row"]).sum()))
print("Duplicate group keys:", int(df.duplicated(subset=BUSINESS_KEY).sum()))

,attribute,nulls
0,year,0
1,month,0
2,primary_sector,0
3,registration_status,0
4,municipality,0
5,taxpayer_count,0
6,investment_purchases_imports_without_vat,0
7,investment_purchases_imports_nondeductible_vat,0
8,investment_imports_18pct,0
9,investment_imports_8pct,0


Unclassified sector: 4,484 (3.61%)
Duplicate source rows: 0
Duplicate group keys: 0


No literal nulls were found in the 12 source attributes. `Mungon aktiviteti` is an **unclassified sector**, not a blank cell: those records are retained and flagged rather than assigned an invented sector.

## 5. Data quality: zeros and negative amounts

In [5]:
money_summary = pd.DataFrame({
    "zeros": df[MONEY].eq(0).sum(),
    "negative_cells": df[MONEY].lt(0).sum(),
}).reset_index().rename(columns={"index": "attribute"})
display(money_summary)

any_negative = df[MONEY].lt(0).any(axis=1)
all_zero = df[MONEY].eq(0).all(axis=1)
print("Rows with at least one negative:", int(any_negative.sum()))
print("Rows with all six amounts equal to zero:", int(all_zero.sum()))

negative_impact = pd.DataFrame({
    "negative_sum": df[MONEY].where(df[MONEY].lt(0), 0).sum(),
    "net_sum": df[MONEY].sum(),
}).reset_index().rename(columns={"index": "attribute"})
display(negative_impact.round(2))

,attribute,zeros,negative_cells
0,investment_purchases_imports_without_vat,104785,36
1,investment_purchases_imports_nondeductible_vat,113111,13
2,investment_imports_18pct,105878,3
3,investment_imports_8pct,121583,0
4,domestic_investment_purchases_18pct,85656,20
5,domestic_investment_purchases_8pct,113245,10


Rows with at least one negative: 79
Rows with all six amounts equal to zero: 77955


,attribute,negative_sum,net_sum
0,investment_purchases_imports_without_vat,-4976519.20,2.975787e+09
1,investment_purchases_imports_nondeductible_vat,-906050.25,1.008511e+09
2,investment_imports_18pct,-73987.54,1.016100e+09
3,investment_imports_8pct,0.00,3.081882e+07
4,domestic_investment_purchases_18pct,-266347.69,4.393589e+09
5,domestic_investment_purchases_8pct,-4800.53,8.704302e+07


**Decision:** keep zeros and negative amounts as reported. Their interpretation is not fully confirmed; changing them to positive values or imputing them would alter the source data. Review their effects separately in later analyses.

## 6. Cleaning: what changed?

In [6]:
cleaning_changes = []
for col in CATEGORIES:
    original = df[f"{col}_raw"].astype("string")
    cleaned = df[col].astype("string")
    changed = original.ne(cleaned).fillna(False)
    cleaning_changes.append({"attribute": col, "changed_cells": int(changed.sum())})
display(pd.DataFrame(cleaning_changes))
print("Rows retained:", f"{len(df):,}")
print("Negative rows marked for review:", int(df["review_negative_monetary"].sum()))

,attribute,changed_cells
0,primary_sector,0
1,registration_status,1414
2,municipality,0


Rows retained: 124,262
Negative rows marked for review: 79


Whitespace at the start or end of categorical labels was trimmed. Unclassified sectors, zeros and negative amounts were not deleted. The original category values remain available in `_raw` columns for comparison.

## 7. Final validation

In [7]:
status_path = root / "data/processed/common_validation.json"
if not status_path.exists():
    raise FileNotFoundError("Validation report not found. Run the processing pipeline first.")
status = json.loads(status_path.read_text(encoding="utf-8"))
print("Mandatory checks:", status.get("mandatory_status"))
print("Records in validation report:", status.get("records"))
print("Checks:", status.get("checks"))
print("Domain status:", status.get("domain_status"))
assert status.get("mandatory_status") == "PASS", "Mandatory validation did not pass."
assert status.get("records") == len(df), "Row count differs from validation report."
assert df[MONEY].notna().all().all(), "Unexpected null in monetary columns."
print("Dataset ready for analytical pipelines (with documented domain limitations).")

Mandatory checks: PASS
Records in validation report: 124262
Checks: {'PASS': 12, 'REVIEW': 1, 'REVIEW/UNRESOLVED': 6}
Domain status: REVIEW/UNRESOLVED
Dataset ready for analytical pipelines (with documented domain limitations).


### Conclusion

The common dataset contains the cleaned and integrated reporting groups. It can be used as input for aggregation, sampling and feature engineering.